# MODELO ARIMA – PROJEÇÃO DA TAXA SELIC

## Modelagem da Taxa de Juros com ARIMA

A **SELIC (Sistema Especial de Liquidação e Custódia)** corresponde à taxa básica de juros da economia brasileira, definida pelo Comitê de Política Monetária (COPOM) do Banco Central do Brasil.

Trata-se do principal instrumento de política monetária, influenciando diretamente:

- Nível de atividade econômica  
- Dinâmica inflacionária  
- Oferta de crédito  
- Taxa de câmbio  
- Decisões de consumo e investimento  

Por se tratar de uma série temporal mensal, a SELIC apresenta características específicas:

- Elevada persistência temporal  
- Ajustes graduais (interest rate smoothing)  
- Mudanças associadas a ciclos econômicos  
- Possíveis quebras estruturais decorrentes de alterações no regime de política monetária  

---

## Estrutura da Dinâmica da SELIC

Diferentemente da inflação, a taxa SELIC não apresenta, em regra, forte componente sazonal anual. Sua trajetória é predominantemente determinada por:

- Estratégia de suavização da política monetária  
- Reação sistemática a choques inflacionários  
- Respostas a crises econômicas  
- Mudanças no regime de metas de inflação  

Assim, sua modelagem enfatiza a dependência temporal não sazonal, sendo adequada a utilização de modelos ARIMA.

---

## Justificativa do Modelo ARIMA

O modelo **ARIMA (AutoRegressive Integrated Moving Average)** é apropriado para séries que apresentam:

- Dependência autoregressiva significativa  
- Possível não estacionariedade em nível  
- Ausência de sazonalidade estruturada  

O modelo permite capturar a persistência da taxa de juros e os efeitos de choques passados sobre sua trajetória futura.

---

## Especificação Geral

O modelo ARIMA é definido como:

$ARIMA(p,d,q)$

Onde:
- **p** → ordem autoregressiva (AR)  
  Representa a dependência da taxa SELIC em relação aos seus próprios valores passados.

- **d** → grau de diferenciação  
  Aplicado quando a série apresenta raiz unitária (não estacionariedade).

- **q** → ordem da média móvel (MA)  
  Captura o efeito de choques passados sobre o valor atual da taxa.

---

## Interpretação Econômica dos Componentes

### Componente AR (p)

Reflete a persistência da política monetária.  
Valores elevados de **p** indicam forte inércia na condução da taxa básica de juros, compatível com a estratégia de suavização adotada pelo Banco Central.

### Diferenciação (d)

Necessária quando o teste de Dickey-Fuller Aumentado (ADF) indica presença de raiz unitária.  
A diferenciação remove tendência estocástica, tornando a série estacionária e adequada à modelagem ARIMA.

### Componente MA (q)

Representa o impacto de choques não antecipados na condução da política monetária.  
Captura ajustes temporários decorrentes de surpresas inflacionárias ou choques macroeconômicos.

---

## Critérios para Escolha dos Parâmetros

A definição da ordem ótima do modelo será baseada em:

- Teste de estacionariedade (ADF – Augmented Dickey-Fuller)  
- Análise das funções de autocorrelação (ACF)  
- Análise das funções de autocorrelação parcial (PACF)  
- Critérios de informação (AIC e BIC)  

O modelo selecionado será aquele que apresentar menor valor de AIC, garantindo equilíbrio entre qualidade de ajuste e parcimônia.

---

## Objetivo

Estimar um modelo estatisticamente consistente para a taxa SELIC, capaz de:

- Capturar sua dinâmica temporal  
- Representar adequadamente sua persistência  
- Gerar projeções futuras coerentes com a estrutura observada da política monetária brasileira  

## TESTE ADF

In [160]:
def testar_estacionariedade(serie, tipo='c'):
    """
    Teste de Estacionariedade via Dickey-Fuller Aumentado (ADF)

    Objetivo:
    Verificar a presença de raiz unitária na série temporal,
    condição que indica não estacionariedade.

    Fundamentação Econométrica:
    O teste ADF (Augmented Dickey-Fuller) amplia o teste de
    Dickey-Fuller tradicional ao incluir defasagens da variável
    dependente, corrigindo autocorrelação serial nos resíduos.

    Hipóteses do Teste:
    H0: A série possui raiz unitária (não estacionária)
    H1: A série não possui raiz unitária (estacionária)

    Parâmetros:
    serie : pandas.Series
        Série temporal a ser testada.

    tipo : str
        Especificação da regressão determinística:
        'c'  → constante (drift)
        'ct' → constante + tendência
        'nc' → sem constante

    Interpretação:
    Se p-valor < 0.05 → Rejeita-se H0 → Série estacionária.
    Caso contrário → Não se rejeita H0 → Série não estacionária.

    Retorno:
    True  → Série estacionária
    False → Série não estacionária
    """

    # Remove valores ausentes para evitar erro no teste
    serie_limpa = serie.dropna()
    
    # Executa o teste ADF com a especificação escolhida
    resultado = adfuller(serie_limpa, regression=tipo)
    
    # Extração dos principais resultados do teste
    estatistica = resultado[0]   # Estatística do teste
    p_valor = resultado[1]       # p-valor associado
    lags = resultado[2]          # Número de defasagens utilizadas
    n_obs = resultado[3]         # Número de observações consideradas
    
    # Exibição dos resultados
    print(f"Tipo de regressão: {tipo}")
    print(f"Estatística ADF: {estatistica}")
    print(f"p-valor: {p_valor}")
    print(f"Lags utilizados: {lags}")
    print(f"Número de observações: {n_obs}")
    
    # Regra de decisão ao nível de 5% de significância
    if p_valor < 0.05:
        print("Conclusão: Rejeita-se H0 → Série estacionária")
        return True
    else:
        print("Conclusão: Não se rejeita H0 → Série não estacionária")
        return False

## Definição dos Parâmetros do SARIMA
$$
ARIMA(p,d,q)
$$

# Definição de *d* (Diferenciação Não Sazonal)

## Objetivo
Tornar a série estacionária em nível, eliminando tendência estocástica ou presença de raiz unitária.

A estacionariedade implica:

- Média constante ao longo do tempo  
- Variância constante  
- Estrutura de dependência temporal estável  

---

## Procedimento: Teste ADF (Augmented Dickey-Fuller)

O teste ADF é utilizado para verificar a presença de raiz unitária.

### Hipóteses:

- **H₀:** A série possui raiz unitária (não estacionária)  
- **H₁:** A série é estacionária  

### Regra de decisão:

- Se **p-valor < 0,05** → rejeita-se H₀ → série estacionária → **d = 0**
- Se **p-valor > 0,05** → não se rejeita H₀ → série não estacionária → aplica-se diferenciação → **d = 1**

---

## Diferenciação de Primeira Ordem

A transformação é dada por:

$\Delta Y_t = Y_t - Y_{t-1}$

Se após a primeira diferença a série se tornar estacionária:

$d = 1$

Em séries macroeconômicas, raramente é necessário utilizar:

$d = 2$

Na maioria dos casos:

$d = 0 \quad \text{ou} \quad d = 1$

---

# Definição de *p* e *q* (Estrutura Dinâmica)

O modelo:

$ARIMA(p,d,q)$

captura a dinâmica temporal da série por meio de dois componentes fundamentais:

- **AR (p)** → dependência dos valores passados  
- **MA (q)** → dependência dos erros (choques) passados  

---

# Identificação via ACF e PACF
## Identificação de *p* (Componente AR)

Baseia-se na análise da **PACF (Partial Autocorrelation Function)**:

- Pico significativo no lag 1 → **p = 1**
- Picos significativos até lag 2 → **p = 2**
- Corte brusco na PACF → indica presença de componente AR

**Regra prática:**  
Corte na PACF → estrutura autoregressiva.

---

## Identificação de *q* (Componente MA)

Baseia-se na análise da **ACF (Autocorrelation Function)**:

- Pico significativo no lag 1 → **q = 1**
- Picos significativos até lag 2 → **q = 2**
- Corte brusco na ACF → indica presença de componente MA

**Regra prática:**  
Corte na ACF → estrutura de média móvel.

---

# Ausência de Sazonalidade

No modelo ARIMA tradicional:

- Não há componente sazonal \((P, D, Q)\)
- Não há periodicidade explícita \( s \)

Utiliza-se ARIMA quando:

- A série não apresenta padrão sazonal recorrente  
- A ACF não evidencia picos sistemáticos em múltiplos fixos  
- Não há indícios de raiz unitária sazonal  


In [ ]:
def ajustar_arima_auto(serie, d=1):
    """
    Ajuste automático do modelo ARIMA(p,d,q)
    com seleção baseada no Critério de Informação de Akaike (AIC).

    Fundamentação Econométrica:
    O modelo ARIMA (AutoRegressive Integrated Moving Average)
    é amplamente utilizado para modelagem de séries temporais
    não sazonais, especialmente em contextos macroeconômicos
    como taxas de juros (ex.: SELIC).

    A estrutura do modelo é dada por:

        ARIMA(p, d, q)

    Onde:
        p → ordem do componente autoregressivo (AR),
             capturando dependência temporal defasada.

        d → número de diferenciações aplicadas à série,
             utilizadas para tornar a série estacionária
             quando há presença de raiz unitária.

        q → ordem do componente de média móvel (MA),
             modelando choques passados (erros defasados).

    Estratégia de Seleção do Modelo:
    A escolha do melhor modelo é realizada por busca em grade
    (grid search) sobre combinações de (p,q), mantendo d fixo,
    com base no menor Critério de Informação de Akaike (AIC).

    O AIC é definido como:

        AIC = -2 log(L) + 2k

    Onde:
        L → função de verossimilhança do modelo estimado
        k → número de parâmetros estimados

    O AIC penaliza modelos excessivamente complexos,
    promovendo parcimônia e bom ajuste simultaneamente.

    Critério de Validação:
    Apenas modelos que convergirem numericamente
    (verificação via resultado.mle_retvals['converged'])
    são considerados válidos.

    Retorno:
    melhor_modelo → objeto ajustado do SARIMAX
    df_resultados → DataFrame com ranking dos modelos por AIC
    """

    # Espaço de busca para p e q (ordens AR e MA)
    p = range(0, 4)   # 0,1,2,3
    q = range(0, 4)   # 0,1,2,3

    parametros = list(itertools.product(p, q))

    melhor_aic = float("inf")
    melhor_modelo = None
    melhor_param = None

    resultados = []

    # Busca exaustiva nas combinações (p,q)
    for param in parametros:
        p_, q_ = param

        try:
            modelo = SARIMAX(
                serie,
                order=(p_, d, q_),
                trend='n',  # sem termo determinístico (avaliado separadamente)
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            # Ajuste via máxima verossimilhança
            resultado = modelo.fit(method='powell', disp=False)

            # Verifica convergência do algoritmo numérico
            if resultado.mle_retvals['converged']:

                aic = resultado.aic
                bic = resultado.bic

                resultados.append({
                    'p': p_,
                    'd': d,
                    'q': q_,
                    'AIC': aic,
                    'BIC': bic
                })

                # Seleção do menor AIC
                if aic < melhor_aic:
                    melhor_aic = aic
                    melhor_modelo = resultado
                    melhor_param = (p_, d, q_)

        except:
            # Ignora modelos que apresentem erro numérico
            continue

    df_resultados = pd.DataFrame(resultados).sort_values("AIC")

    print("Melhor modelo encontrado:")
    print(f"ARIMA{melhor_param}")
    print(f"AIC: {melhor_aic}")
    print("\nResumo do modelo:")
    print(melhor_modelo.summary())

    return melhor_modelo, df_resultados

## Previsão da Selic

In [ ]:
def projetar_arima(modelo, serie, ano_final=2030, mes_final=1):
    
    """
    Projeção da Taxa SELIC até data específica (ano e mês).

    Fundamentação Metodológica:
    A projeção é realizada com base em um modelo ARIMA(p,d,q)
    previamente estimado por Máxima Verossimilhança.

    O horizonte de previsão é calculado de forma precisa,
    considerando a diferença exata em meses entre a última
    observação da amostra e a data final desejada.

    Isso garante coerência com a frequência mensal da série,
    evitando superestimação ou subestimação do período projetado.

    Parâmetros:
    modelo     → objeto ajustado do SARIMAX (ARIMA estimado)
    serie      → série temporal utilizada na estimação
    ano_final  → ano limite da projeção
    mes_final  → mês limite da projeção

    Retorno:
    DataFrame contendo:
        - selic_prevista (valor esperado)
        - limite_inferior (IC 95%)
        - limite_superior (IC 95%)

    Observação:
    A projeção é puramente estatística (inercial),
    não incorporando variáveis exógenas ou hipóteses
    estruturais sobre política monetária futura.
    """
    # Identificação da última data disponível na amostra
    # Assume-se que o índice da série seja do tipo DateTimeIndex
    ultima_data = serie.index[-1]

    # Construção da data final desejada
    # Define-se o primeiro dia do mês como referência temporal
    data_final = pd.Timestamp(year=ano_final, month=mes_final, day=1)

    # Cálculo do número exato de passos (meses) à frente
    # Fórmula:
    # (diferença em anos * 12) + diferença em meses
    passos = (data_final.year - ultima_data.year) * 12 + \
             (data_final.month - ultima_data.month)

    # Verificação de consistência temporal
    if passos <= 0:
        raise ValueError("A data final deve ser posterior à última observação.")

    # Geração da previsão fora da amostra
    # get_forecast calcula:
    # - média prevista
    # - intervalo de confiança (default: 95%)
    previsao = modelo.get_forecast(steps=passos)

    # Valor esperado da previsão
    media_prevista = previsao.predicted_mean

    # Intervalos de confiança
    intervalo = previsao.conf_int()
    
    # Organização dos resultados em DataFrame estruturado
    df_previsao = media_prevista.to_frame(name='selic_prevista')
    df_previsao['limite_inferior'] = intervalo.iloc[:, 0]
    df_previsao['limite_superior'] = intervalo.iloc[:, 1]

    return df_previsao

## Plotar a Previsão

In [183]:
def plotar_projecao(df_mensal, df_previsao, titulo_modelo="ARIMA"):
    """
    Visualização da Série Histórica e Projeção da Taxa SELIC

    Fundamentação Metodológica:
    O gráfico apresenta a trajetória histórica da taxa SELIC
    juntamente com a projeção fora da amostra gerada pelo
    modelo ARIMA(p,d,q) estimado.

    Componentes do Gráfico:

        1. Série histórica observada (dados reais).
        2. Trajetória projetada pelo modelo.
        3. Intervalo de confiança de 95%.
        4. Marco temporal de separação entre amostra e previsão.

    Interpretação Econométrica:

    - A linha contínua representa os valores efetivamente observados.
    - A linha tracejada representa o valor esperado da previsão,
      condicional à informação disponível até o último período da amostra.
    - A área sombreada corresponde ao intervalo de confiança
      baseado na variância do erro de previsão.
    - O alargamento progressivo do intervalo decorre da
      propagação da incerteza inerente ao processo estocástico
      modelado pelo ARIMA.

    Observação:
    A projeção é de natureza inercial, isto é,
    baseada exclusivamente na dinâmica estatística passada,
    sem incorporação de variáveis exógenas ou hipóteses
    estruturais sobre política monetária futura.
    """

    # Garantir formato adequado
    df_mensal.index = pd.to_datetime(df_mensal.index)
    df_previsao.index = pd.to_datetime(df_previsao.index)

    # Garantia de tipo numérico para evitar erros de plotagem
    df_mensal['valor'] = df_mensal['valor'].astype(float)
    df_previsao = df_previsao.astype(float)

    # Configuração da figura
    plt.figure(figsize=(12, 6))

    # Série histórica
    plt.plot(
        df_mensal.index,
        df_mensal['valor'],
        label='Histórico Observado',
        linewidth=2
    )

    # Trajetória projetada
    plt.plot(
        df_previsao.index,
        df_previsao['selic_prevista'],
        linestyle='--',
        linewidth=2,
        label=f'Projeção {titulo_modelo}'
    )

    # Intervalo de confiança
    plt.fill_between(
        df_previsao.index,
        df_previsao['limite_inferior'],
        df_previsao['limite_superior'],
        alpha=0.3,
        label='Intervalo de Confiança (95%)'
    )

    # Linha vertical indicando início da projeção
    plt.axvline(
        df_mensal.index[-1],
        linestyle=':',
        label='Início da Projeção'
    )

    # Elementos gráficos finais
    plt.title(f'Projeção da Taxa SELIC via {titulo_modelo}')
    plt.xlabel('Ano')
    plt.ylabel('Taxa (%)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()